# Notebook 1 — Data Understanding & SQL-style Analysis

Notebook này đã được chỉnh để chạy trực tiếp với các file bạn tải lên:
- `MolaDatabase.invoices.json`
- `MolaDatabase.invoice_items.json`
- `join_3NF_result.xlsx`

Mục tiêu:
1. Đọc đúng dữ liệu hóa đơn và chi tiết hóa đơn.
2. Chuẩn hóa các cột quan trọng theo tên dễ dùng.
3. Mô phỏng logic `CASE WHEN ... LIKE` trong SQL để phân loại dịch vụ.
4. Nối dữ liệu theo khóa `unique_key` ↔ `invoice_unique_key`.
5. Tạo bảng tổng hợp theo hóa đơn để dùng tiếp cho các notebook sau.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)

# Thư mục đang chạy notebook
BASE_DIR = Path.cwd()

# Một số nơi thường gặp mà bạn có thể đặt dữ liệu
SEARCH_DIRS = [
    BASE_DIR,
    BASE_DIR / "data",
    BASE_DIR / "dataset",
    BASE_DIR / "datasets",
    BASE_DIR / "raw_data",
]

def resolve_path(*candidates):
    for folder in SEARCH_DIRS:
        for name in candidates:
            p = folder / name
            if p.exists():
                return p
    raise FileNotFoundError(
        "Không tìm thấy file dữ liệu. Hãy đặt file cùng thư mục với notebook hoặc trong data/.\n"
        f"Đã tìm các tên: {candidates}\n"
        f"Đã dò các thư mục: {[str(p) for p in SEARCH_DIRS]}"
    )

def load_json_dataframe(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return pd.json_normalize(data)

def categorize_service(item_name: str) -> str:
    text = str(item_name).lower()
    if "kiểm định" in text:
        return "Inspection"
    if "giám định" in text:
        return "Survey"
    if "hiệu chuẩn" in text:
        return "Calibration"
    if "thử nghiệm" in text or "phân tích" in text:
        return "Testing"
    return "Other"


In [ ]:
# Đọc dữ liệu từ các file thật của dự án
invoices_path = resolve_path("MolaDatabase.invoices.json")
items_path = resolve_path("MolaDatabase.invoice_items.json")
analytics_path = resolve_path("MolaDatabase.invoice_analytics.json")
excel_path = resolve_path("join_3NF_result.xlsx")

df_invoices_raw = load_json_dataframe(invoices_path)
df_items_raw = load_json_dataframe(items_path)
df_analytics_raw = load_json_dataframe(analytics_path)

print("Invoices path:", invoices_path)
print("Items path:", items_path)
print("Analytics path:", analytics_path)
print("Excel path:", excel_path)
print("-" * 80)
print("Invoices raw shape:", df_invoices_raw.shape)
print("Invoice items raw shape:", df_items_raw.shape)
print("Invoice analytics raw shape:", df_analytics_raw.shape)

display(df_invoices_raw.head(3))
display(df_items_raw.head(3))
display(df_analytics_raw.head(3))


In [ ]:
# Chuẩn hóa tên cột và kiểu dữ liệu
invoice_rename_map = {
    "unique_key": "invoice_unique_key",
    "issue_date": "invoice_date",
    "buyer.name": "buyer_name",
    "buyer.tax_code": "buyer_tax_code",
    "financial_summary.subtotal_before_tax": "subtotal_before_tax",
    "financial_summary.total_tax": "total_tax",
    "financial_summary.total_amount": "total_amount",
    "processing_info.status": "invoice_status",
    "processing_info.payment_method": "payment_method",
}

item_rename_map = {
    "subtotal": "item_subtotal",
}

analytics_rename_map = {
    "date": "analytics_date",
    "total_revenue": "analytics_total_revenue",
    "total_tax": "analytics_total_tax",
    "total_items": "analytics_total_items",
    "total_invoices": "analytics_total_invoices",
}

df_invoices = df_invoices_raw.rename(columns=invoice_rename_map).copy()
df_items = df_items_raw.rename(columns=item_rename_map).copy()
df_analytics = df_analytics_raw.rename(columns=analytics_rename_map).copy()

if "invoice_date" in df_invoices.columns:
    df_invoices["invoice_date"] = pd.to_datetime(df_invoices["invoice_date"], errors="coerce")

if "analytics_date" in df_analytics.columns:
    df_analytics["analytics_date"] = pd.to_datetime(df_analytics["analytics_date"], errors="coerce")

numeric_invoice_cols = [
    "subtotal_before_tax", "total_tax", "total_amount", "item_count"
]
for col in numeric_invoice_cols:
    if col in df_invoices.columns:
        df_invoices[col] = pd.to_numeric(df_invoices[col], errors="coerce")

numeric_item_cols = [
    "quantity", "unit_price", "item_subtotal", "tax_amount", "discount", "sequence"
]
for col in numeric_item_cols:
    if col in df_items.columns:
        df_items[col] = pd.to_numeric(df_items[col], errors="coerce")

numeric_analytics_cols = [
    "month", "year", "analytics_total_invoices", "analytics_total_items",
    "analytics_total_revenue", "analytics_total_tax"
]
for col in numeric_analytics_cols:
    if col in df_analytics.columns:
        df_analytics[col] = pd.to_numeric(df_analytics[col], errors="coerce")

base_invoice_cols = [
    "invoice_unique_key", "invoice_number", "invoice_date", "buyer_name", "buyer_tax_code",
    "subtotal_before_tax", "total_tax", "total_amount", "payment_method", "invoice_status", "item_count"
]

print("Các cột chính của invoices:")
print(base_invoice_cols)
display(df_invoices[base_invoice_cols].head())

print("Các cột chính của invoice_items:")
display(df_items[["invoice_unique_key", "item_name", "quantity", "unit_price", "item_subtotal", "item_type"]].head())

print("Các cột chính của analytics:")
display(df_analytics.head())


In [ ]:
# Phân loại dịch vụ theo mô tả item_name (mô phỏng SQL CASE WHEN)
df_items["service_category"] = df_items["item_name"].apply(categorize_service)

service_summary = (
    df_items[df_items["item_type"].eq("product_service")]
    .groupby("service_category", as_index=False)
    .agg(
        total_rows=("item_name", "count"),
        total_quantity=("quantity", "sum"),
        total_item_subtotal=("item_subtotal", "sum"),
    )
    .sort_values("total_item_subtotal", ascending=False)
)

print("Bảng tổng hợp nhóm dịch vụ:")
display(service_summary)

print("Ví dụ phân loại 10 dòng đầu:")
display(df_items[["item_name", "service_category", "quantity", "unit_price", "item_subtotal"]].head(10))


In [ ]:
# Nối dữ liệu item-level với invoice-level theo khóa 3NF
joined = df_items.merge(
    df_invoices[base_invoice_cols],
    on="invoice_unique_key",
    how="left",
    validate="many_to_one",
)

print("Joined shape:", joined.shape)
print("Số invoice key trong bảng joined:", joined["invoice_unique_key"].nunique())
print("Số dòng không khớp total_amount:", joined["total_amount"].isna().sum())

display(
    joined[
        [
            "invoice_unique_key", "item_name", "service_category",
            "buyer_name", "invoice_date", "total_amount"
        ]
    ].head(10)
)


In [ ]:
# Tổng hợp lên mức hóa đơn để dùng cho EDA / modeling
product_items = joined[joined["item_type"].eq("product_service")].copy()

invoice_level = (
    product_items
    .groupby("invoice_unique_key", as_index=False)
    .agg(
        invoice_date=("invoice_date", "first"),
        invoice_number=("invoice_number", "first"),
        buyer_name=("buyer_name", "first"),
        buyer_tax_code=("buyer_tax_code", "first"),
        payment_method=("payment_method", "first"),
        invoice_status=("invoice_status", "first"),
        total_amount=("total_amount", "first"),
        subtotal_before_tax=("subtotal_before_tax", "first"),
        total_tax=("total_tax", "first"),
        item_count=("item_count", "first"),
        total_quantity=("quantity", "sum"),
        avg_unit_price=("unit_price", "mean"),
        max_item_subtotal=("item_subtotal", "max"),
        service_category_mode=("service_category", lambda s: s.mode().iat[0] if not s.mode().empty else "Other"),
    )
)

invoice_level["month"] = invoice_level["invoice_date"].dt.to_period("M").astype(str)
invoice_level["quarter"] = invoice_level["invoice_date"].dt.quarter
invoice_level["year"] = invoice_level["invoice_date"].dt.year
invoice_level["buyer_name_short"] = invoice_level["buyer_name"].fillna("Unknown").str.slice(0, 40)

prepared_path = BASE_DIR / "prepared_invoice_dataset.csv"
invoice_level.to_csv(prepared_path, index=False, encoding="utf-8-sig")

print(f"Đã lưu dataset tổng hợp: {prepared_path.name}")
print("Invoice-level shape:", invoice_level.shape)
display(invoice_level.head())


In [ ]:
# Đối chiếu nhanh với file Excel join_3NF_result.xlsx
xls = pd.ExcelFile(excel_path)
print("Các sheet hiện có:", xls.sheet_names)

joined_excel = pd.read_excel(excel_path, sheet_name="Joined_Per_Item")
print("Excel Joined_Per_Item shape:", joined_excel.shape)

cols_to_show = [c for c in ["invoice_unique_key", "item_name", "issue_date", "total_amount", "payment_method"] if c in joined_excel.columns]
display(joined_excel[cols_to_show].head())
